# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print("Dataset loaded:")
print(f"Name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

# You can examine other metadata fields as needed
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Every record set, field, and column is referenced ONLY by its `@id` as required. This ensures data is referenced consistently and in compliance with Croissant best practices.

In [ ]:
# Let's enumerate all record sets (`@id`s), their fields, and columns
print("Available record sets and fields:")
record_sets = []
for rs in dataset.metadata.recordSet:
    print(f'RecordSet `@id`: {rs["@id"]}')
    record_sets.append(rs["@id"])
    if hasattr(rs, 'field'):
        print("  Fields:")
        for field in rs.field:
            print(f"   - Field @id: {field['@id']}; Name: {field.get('name', '')}")
    if hasattr(rs, 'column'):
        print("  Columns:")
        for col in rs.column:
            print(f"   - Column @id: {col['@id']}; Name: {col.get('name', '')}")

if not record_sets:
    print("No record sets were found in metadata. Please check the Croissant schema for details.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example extraction of all record sets into DataFrames, referenced by their `@id`
dataframes = {}
if not record_sets:
    print("No record sets were found; please check the metadata.")
else:
    for record_set_id in record_sets:
        print(f"Loading records for record set: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for {record_set_id}")
            print("Fields:", df.columns.tolist())
            print(df.head())
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

# Choose a record set to focus EDA (the first, if any)
focus_record_set = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All EDA steps work on `@id` referenced fields or columns only.

In [ ]:
import numpy as np

if focus_record_set and focus_record_set in dataframes:
    df = dataframes[focus_record_set]
    print(f"Columns in the DataFrame for {focus_record_set}:")
    print(df.columns.tolist())

    # Attempt to find a numeric field (`@id`) for demonstration
    # We'll try common names (case-insensitive), else pick the first float/int column
    numeric_field_candidates = [col for col in df.columns if 'count' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower() or 'mw' in col.lower() or 'pi' in col.lower()]
    if not numeric_field_candidates:
        # Try to select by dtype
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field_candidates.append(col)

    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]  # Take the first one
        print(f"Using numeric field: {numeric_field} for EDA")
    else:
        numeric_field = None
        print("No numeric fields found for EDA.")

    # Select another field for grouping (e.g. protein description or accession)
    group_field_candidates = [col for col in df.columns if 'group' in col.lower() or 'description' in col.lower() or 'accession' in col.lower()]
    group_field = group_field_candidates[0] if group_field_candidates else None

    if numeric_field is not None:
        # Drop missing
        filtered_df = df.dropna(subset=[numeric_field])
        # Filter for values greater than median (or 10 if that makes sense)
        threshold = 10 if filtered_df[numeric_field].max() > 10 else filtered_df[numeric_field].median()
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered to records where {numeric_field} > {threshold}, count: {len(filtered_df)}")

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())
    else:
        print('No numeric field found for EDA.')
else:
    print('No focus record set DataFrame available for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, we show a histogram and a boxplot (if possible) on the chosen numeric field referenced by its `@id`.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if focus_record_set and focus_record_set in dataframes and numeric_field is not None:
    df = dataframes[focus_record_set]
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    df[numeric_field].dropna().hist(bins=30, edgecolor='k')
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field)
    plt.title(f'Boxplot of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough data to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset was loaded successfully from its Croissant schema using `mlcroissant` by URL.
* Data contains record sets and fields accessible by their `@id`, making data curation and referencing robust.
* We filtered, normalized, grouped, and visualized key numeric fields by referencing them dynamically by their `@id`.

**Continue exploring further with domain-specific questions, or adapt this template for similar Croissant datasets.**